In [1]:
"""
reconcile_column_counts.py
===========================

Traces the aggregate "moment column" count through every stage of the
pipeline, from the 2,724-column figure in the Data Cleaning chapter
(Stage_2 -> Stage_3_Cleaning/01_apply_exclusions ->
Stage_3_Cleaning/04_normalise, giving the reported 2,437-column figure)
through to the final 1,699-column feature set claimed at the start of
that same chapter.

WHY THIS EXISTS
---------------
The write-up says:

    "Aggregate moments table, by bucket: 287 of 2,724 columns (10.5%)
    were dropped in total ... After exclusion, 2,437 of the original
    2,724 aggregate moment columns remain. This is the process that
    produces the 1,699-factor, 331-subtheme dataset introduced at the
    start of this chapter."

2,437 != 1,699. Worse: 2,437 is not even a file the pipeline ever
writes. Stage_3_Cleaning/04_normalise.ipynb saves its z-scored parquets
BEFORE computing the structural/R4/R5 drops, and decisions_aggregate.csv
only records a verdict per column -- it never re-saves a trimmed
parquet. 2,437 is a hypothetical: "what would survive if you stopped
after rules S/R0-R6 alone."

The real pipeline keeps going. Three more removal passes happen after
that point, none of which are described in the "Exclusion Rules and
Results" section:

    1. Stage_4_Assembly/01_apply_union.ipynb (PASS 1 + PASS 2), which
       applies, all at once:
         - union_drop_list.csv: base factors dropped because their
           cap-weighted MEAN failed a rule in either the aggregate or
           the panel pipeline
         - MANUAL_DROP: 13 named factors, dropped by hand, never
           assessed by S/R0-R6 at all
         - identity enforcement: stock-level base factors present in
           only ONE of {aggregate, panel}, removed from both

    2. Stage_4_Assembly/07_drop_binaries.ipynb, which removes the 5
       binary regime indicators (vix_above_20, vix_above_30,
       curve_inverted_2y10y, curve_inverted_3m10y, credit_stress).
       These were exempt from S/R0-R6 (SKIP_ZSCORE in 04_normalise),
       so they never even appear in decisions_aggregate.csv.

This script recomputes the ACTUAL column counts at every real
checkpoint directly from the parquet/csv files the pipeline writes, so
you can either correct the write-up with real numbers or catch a
regression if the pipeline no longer matches what's described.

USAGE
-----
    python reconcile_column_counts.py --root "C:/path/to/Data/Data_Collection/Final"

If --root is omitted, the script tries a few likely relative locations
(as if run from inside Code/Data_Merging/<some Stage_X folder>/).

Every checkpoint is wrapped so a missing file is reported and skipped
rather than crashing the whole run -- useful if you've only
regenerated part of the pipeline since making a change.
"""

import argparse
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# ──────────────────────────────────────────────────────────────────────────
# REFERENCE NUMBERS FROM THE WRITE-UP
# Hardcoded so this script can flag drift between what was written and
# what the pipeline currently produces.
# ──────────────────────────────────────────────────────────────────────────

CHAPTER = {
    "total_moment_columns_before_rules": 2724,
    "moment_columns_dropped_by_rules":   287,
    "moment_columns_after_rules":        2437,   # <- the contested number
    "final_aggregate_features":          1699,   # <- claimed to be "the same process"
    "final_aggregate_subthemes":         331,
    "final_means_only_features":         574,
    "final_means_only_subthemes":        128,
    "base_factors_dropped_macro":        37,
    "base_factors_dropped_stock":        39,
}

CHAPTER_MACRO_BY_RULE = {"S": 18, "R0": 0, "R1": 0, "R2": 10,
                          "R3": 0, "R4": 8, "R5": 5, "R6": 6}

MOMENTS = ("_cwmean", "_cwstd", "_cwskew", "_cwkurt", "_spread")

# The three tables that sum to the chapter's "2,724 aggregate moment
# columns". weekly_raw counts in full because macro factors never have
# moments -- one column always stands for one factor, whether it's a
# "mean", a "std", or a macro raw level.
MOMENT_TABLES = ["agg_market_daily_full_moments", "weekly_raw",
                  "agg_market_monthly_full_moments"]

# Base-factor-level grouping used by the pipeline's own
# Stage_3_Cleaning/06_which_factors_dropped.ipynb: exactly one column
# per base factor in these tables, so counting dropped ROWS equals
# counting dropped FACTORS.
MEANS_TABLES = {"agg_market_daily_means", "agg_market_monthly_means", "weekly_raw"}

META_BY_STEM = {
    "agg_market_daily_means":          {"date", "target_daily_return"},
    "agg_market_daily_full_moments":   {"date", "target_daily_return"},
    "weekly_raw":                      {"date"},
    "agg_market_monthly_means":        {"date", "target_monthly_return"},
    "agg_market_monthly_full_moments": {"date", "target_monthly_return"},
    "panel_stock_daily_engineered":    {"permno", "date", "dlyret", "dlycap"},
    "panel_stock_monthly_engineered":  {"permno", "date", "month_end_cap",
                                        "month_end_price"},
    "agg_means":         {"date", "target_daily_return", "minret_5d_pct", "y_binary"},
    "agg_full_moments":  {"date", "target_daily_return", "minret_5d_pct", "y_binary"},
    "panel":             {"permno", "date", "dlyret", "dlycap",
                           "minret_5d_pct", "y_binary"},
}

BINARIES = {"vix_above_20", "vix_above_30", "curve_inverted_2y10y",
            "curve_inverted_3m10y", "credit_stress"}

# The 13 factors Stage_4_Assembly/01_apply_union.ipynb drops by hand,
# never assessed by any of rules S/R0-R6.
MANUAL_DROP = {
    "stock_skew_chg_5d",
    "CBOperProf", "NetPayoutYield", "AOP", "GrSaleToGrOverhead", "GP",
    "MomSeason16YrPlus", "GrSaleToGrInv", "MomOffSeason16YrPlus",
    "DivYieldST", "fgr5yrLag", "MomSeason11YrPlus", "InvestPPEInv",
}


# ──────────────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────────────

def base_factor_map(columns):
    """Column -> base factor. Strips a moment suffix ONLY when the matching
    _cwmean column is also present in `columns`, so macro names that
    legitimately end in e.g. '_spread' (bull_bear_spread, vix_term_spread,
    ...) are left whole. Mirrors the logic described across your own
    Stage_4_Assembly / Stage_3_Cleaning notebooks."""
    colset = set(columns)
    out = {}
    for c in columns:
        base = c
        for suf in MOMENTS:
            if c.endswith(suf):
                cand = c[: -len(suf)]
                if f"{cand}_cwmean" in colset:
                    base = cand
                break
        out[c] = base
    return out


def cols_of(path: Path):
    """Column names of a parquet file, schema-only (no data load). None if
    the file doesn't exist."""
    if not path.exists():
        return None
    return list(pq.read_schema(path).names)


def n_features(path: Path, stem: str):
    """(total_columns, feature_columns) for a parquet file, or (None, None)
    if missing. 'feature' excludes the meta/id/target columns for that stem."""
    cols = cols_of(path)
    if cols is None:
        return None, None
    meta = META_BY_STEM.get(stem, set())
    feats = [c for c in cols if c not in meta]
    return len(cols), len(feats)


def csv_or_none(path: Path, label: str = None):
    if not path.exists():
        print(f"    (missing: {path})")
        return None
    return pd.read_csv(path)


def hdr(title):
    print("\n" + "=" * 96)
    print(title)
    print("=" * 96)


def sub(title):
    print("\n" + "-" * 96)
    print(title)
    print("-" * 96)


# ──────────────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────────────

def main(root: Path):
    STAGE3    = root / "Stage_3_Cleaning"
    NORM_A    = root / "Stage_4_Normalised"
    NORM_P    = root / "Stage_4_Normalised_Panel"
    UNIONED   = root / "Stage_5_Model_Ready" / "01_unioned"
    ASSEMBLED = root / "Stage_5_Model_Ready" / "02_assembled"
    THEMES    = root / "Stage_5_Model_Ready" / "05_themes"

    hdr("RECONCILING '2,437 columns remain' vs 'the 1,699-factor dataset'")
    print(f"""
  The write-up says S/R0-R6 drop 287 of {CHAPTER['total_moment_columns_before_rules']:,}
  aggregate moment columns, leaving {CHAPTER['moment_columns_after_rules']:,}, and calls this
  "the process that produces the {CHAPTER['final_aggregate_features']:,}-factor ... dataset".
  Those numbers are {CHAPTER['moment_columns_after_rules'] - CHAPTER['final_aggregate_features']:,} apart. This walks the
  ACTUAL files the pipeline writes, checkpoint by checkpoint, to show
  where the gap goes.

  Root: {root}
""")

    # cache these across checkpoints so the summary waterfall can use them
    cp2_pre = None
    cp3_rule_only = None
    cp4_post_union = None
    cp5_assembled = None
    cp6_final = None

    # ────────────────────────────────────────────────────────────────
    # CHECKPOINT 1 -- base-factor-level rule counts
    # Reproduces the chapter's "Base factors dropped, by rule" table.
    # ────────────────────────────────────────────────────────────────
    hdr("CHECKPOINT 1 -- base-factor rule counts (S / R0-R6)")

    dec_agg = csv_or_none(NORM_A / "decisions_aggregate.csv")
    dec_pan = csv_or_none(NORM_P / "decisions_panel.csv")

    if dec_agg is not None and dec_pan is not None:
        bmaps = {t: base_factor_map(g["feature"])
                 for t, g in dec_agg.groupby("table_source")}
        dec_agg_bf = dec_agg.copy()
        dec_agg_bf["base_factor"] = [
            bmaps[t][f] for t, f in zip(dec_agg_bf["table_source"], dec_agg_bf["feature"])
        ]

        macro_drop = dec_agg_bf[dec_agg_bf["table_source"].isin(MEANS_TABLES)
                                 & (dec_agg_bf["action"] == "drop")]
        macro_by_rule = macro_drop.drop_duplicates("base_factor")["rule_fired"].value_counts()

        pan_action_col = "final_action" if "final_action" in dec_pan.columns else "action"
        stock_drop = dec_pan[dec_pan[pan_action_col] == "drop"]
        stock_by_rule = stock_drop.drop_duplicates("base_factor")["rule_fired"].value_counts()

        print(f"\n  {'rule':<6} {'macro (this run)':>18} {'macro (chapter)':>18} "
              f"{'stock (this run)':>18}")
        for rule in ["S", "R0", "R1", "R2", "R3", "R4", "R5", "R6"]:
            m = int(macro_by_rule.get(rule, 0))
            s = int(stock_by_rule.get(rule, 0))
            cm = CHAPTER_MACRO_BY_RULE.get(rule, "?")
            flag = "" if str(m) == str(cm) else "   <-- DIFFERS FROM CHAPTER"
            print(f"  {rule:<6} {m:>18} {cm!s:>18} {s:>18}{flag}")

        n_macro_dropped = macro_drop["base_factor"].nunique()
        n_stock_dropped = stock_drop["base_factor"].nunique()
        print(f"\n  Unique base factors dropped -- macro: {n_macro_dropped} "
              f"(chapter says {CHAPTER['base_factors_dropped_macro']}), "
              f"stock: {n_stock_dropped} "
              f"(chapter says {CHAPTER['base_factors_dropped_stock']})")
    else:
        print("  Skipped -- need both decisions_aggregate.csv and decisions_panel.csv")

    # ────────────────────────────────────────────────────────────────
    # CHECKPOINT 2 -- columns entering 04_normalise
    # ────────────────────────────────────────────────────────────────
    hdr("CHECKPOINT 2 -- columns entering Stage_3_Cleaning/04_normalise")

    cp2_pre = 0
    any_missing = False
    for stem in MOMENT_TABLES:
        _, f3 = n_features(STAGE3 / f"{stem}.parquet", stem)
        _, f4 = n_features(NORM_A / f"{stem}.parquet", stem)
        if f3 is None:
            print(f"  {stem:<34} Stage_3_Cleaning file missing")
            any_missing = True
            continue
        cp2_pre += f3
        note = "" if f3 == f4 else f"   <-- Stage_4_Normalised has {f4}, not the same!"
        print(f"  {stem:<34} {f3:>6} feature columns{note}")

    if any_missing:
        cp2_pre = None
    else:
        print(f"\n  Total across the 3 tables: {cp2_pre:,}   "
              f"(chapter says {CHAPTER['total_moment_columns_before_rules']:,})")
        if cp2_pre != CHAPTER["total_moment_columns_before_rules"]:
            print("  ** Does not match. Either the pipeline has changed since "
                  "the chapter was written, or these three tables are not what "
                  "the chapter measured. **")

    # ────────────────────────────────────────────────────────────────
    # CHECKPOINT 3 -- the "2,437" figure: a COUNT, not a FILE
    # ────────────────────────────────────────────────────────────────
    hdr("CHECKPOINT 3 -- rule-only survivor count (the chapter's '2,437')")
    print("""
  This number is never materialised as a parquet file anywhere in the
  pipeline. 04_normalise.ipynb writes the S/R0-R6 VERDICT to
  decisions_aggregate.csv, but the parquet files in Stage_4_Normalised/
  still carry every column, dropped or not. The count below is purely
  action=='keep' rows in that CSV.
""")

    if dec_agg is not None:
        keep = dec_agg[dec_agg["table_source"].isin(MOMENT_TABLES)
                       & (dec_agg["action"] == "keep")]
        cp3_rule_only = len(keep)
        print(f"  action=='keep' rows across the 3 tables: {cp3_rule_only:,}  "
              f"(chapter says {CHAPTER['moment_columns_after_rules']:,})")
        if cp2_pre is not None:
            implied_drop = cp2_pre - cp3_rule_only
            print(f"  implied drops: {implied_drop:,}  "
                  f"(chapter says {CHAPTER['moment_columns_dropped_by_rules']:,})")
    else:
        print("  Skipped -- decisions_aggregate.csv not available")

    # ────────────────────────────────────────────────────────────────
    # CHECKPOINT 4 -- what ACTUALLY gets removed first: rule + union +
    # manual + identity enforcement, all combined, in
    # Stage_4_Assembly/01_apply_union.ipynb
    # ────────────────────────────────────────────────────────────────
    hdr("CHECKPOINT 4 -- Stage_4_Assembly/01_apply_union.ipynb (real removal)")

    rep = csv_or_none(UNIONED / "union_applied_report.csv")
    if rep is not None:
        rep_m = rep[rep["table"].isin(MOMENT_TABLES)]
        sub("union_applied_report.csv, restricted to the 3 moment tables")
        print(rep_m[["table", "by_union", "by_rule", "by_manual",
                      "by_identity", "features_final"]].to_string(index=False))

        cp4_post_union = int(rep_m["features_final"].sum())
        by_union  = int(rep_m["by_union"].sum())
        by_rule   = int(rep_m["by_rule"].sum())
        by_manual = int(rep_m["by_manual"].sum())
        by_ident  = int(rep_m["by_identity"].sum())

        print(f"\n  Totals across the 3 tables:")
        print(f"    dropped by union (base factor's cwmean failed somewhere) : {by_union:>5,}")
        print(f"    dropped by rule alone (a moment failed, cwmean survived)  : {by_rule:>5,}")
        print(f"    dropped by hand (MANUAL_DROP, 13 named factors)          : {by_manual:>5,}")
        print(f"    dropped for identity (present in only one pipeline)      : {by_ident:>5,}")
        print(f"    {'-' * 62}")
        print(f"    columns remaining after this pass                       : {cp4_post_union:>5,}")

        if cp3_rule_only is not None:
            gap_here = cp3_rule_only - cp4_post_union
            total_gap = CHAPTER["moment_columns_after_rules"] - CHAPTER["final_aggregate_features"]
            print(f"\n  This pass alone accounts for {gap_here:,} of the "
                  f"{total_gap:,}-column gap between 2,437 and 1,699 -- none of "
                  f"which is described in the 'Exclusion Rules and Results' "
                  f"section.")
    else:
        print("  Skipped -- union_applied_report.csv not available. For reference:")
        udl = csv_or_none(NORM_A / "union_drop_list.csv")
        if udl is not None:
            print(f"    union_drop_list.csv: {len(udl)} base factors")
        print(f"    MANUAL_DROP (hardcoded in 01_apply_union.ipynb): "
              f"{len(MANUAL_DROP)} factors -> {sorted(MANUAL_DROP)}")

    # ────────────────────────────────────────────────────────────────
    # CHECKPOINT 5 -- merge into agg_full_moments.parquet, and the five
    # binary regime indicators
    # ────────────────────────────────────────────────────────────────
    hdr("CHECKPOINT 5 -- 02_assembled/agg_full_moments.parquet")

    p_asm = ASSEMBLED / "agg_full_moments.parquet"
    cols_asm = cols_of(p_asm)
    if cols_asm is not None:
        meta = META_BY_STEM["agg_full_moments"]
        feats_asm = [c for c in cols_asm if c not in meta]
        cp5_assembled = len(feats_asm)
        present_bin = sorted(BINARIES & set(cols_asm))

        print(f"  agg_full_moments.parquet: {cp5_assembled:,} feature columns")
        if present_bin:
            print(f"  Binary regime indicators STILL present: {present_bin}")
            print(f"  (07_drop_binaries.ipynb has not been run against this copy "
                  f"of the data, or was run against a different copy.)")
        else:
            print(f"  Binary regime indicators absent -- 07_drop_binaries.ipynb "
                  f"has already run.")

        if cp4_post_union is not None:
            merge_delta = cp5_assembled - cp4_post_union
            print(f"\n  Change from Checkpoint 4 ({cp4_post_union:,}) to here "
                  f"({cp5_assembled:,}): {merge_delta:+,}")
            print(f"  Expected ~0 net change from the merge itself (daily + "
                  f"weekly + monthly are simply consolidated onto one daily "
                  f"calendar). A non-zero difference here is mostly the "
                  f"binary-indicator drop (-5).")
    else:
        print(f"  Skipped -- {p_asm} not found")

    # ────────────────────────────────────────────────────────────────
    # CHECKPOINT 6 -- the final, numbered feature set (should be 1,699)
    # ────────────────────────────────────────────────────────────────
    hdr("CHECKPOINT 6 -- Stage_5_Themes final numbered inventory")

    inv = csv_or_none(THEMES / "numbered_classified_moment_inventory_long.csv")
    if inv is not None:
        cp6_final = len(inv)
        n_sub = inv["subtheme_id"].nunique()
        print(f"  numbered_classified_moment_inventory_long.csv: "
              f"{cp6_final:,} rows, {n_sub} subthemes")
        print(f"  Chapter claims: {CHAPTER['final_aggregate_features']:,} factors, "
              f"{CHAPTER['final_aggregate_subthemes']} subthemes")
        flag = "MATCHES" if cp6_final == CHAPTER["final_aggregate_features"] else "DOES NOT MATCH"
        print(f"  -> {flag}")

        sub("Per-theme feature counts (compare against the write-up's table)")
        by_theme = inv.groupby("theme_name")["column"].nunique().sort_values(ascending=False)
        print(by_theme.to_string())
    else:
        print("  Skipped -- run Stage_5_Themes/04_numbering_subthemes.ipynb first")

    # ────────────────────────────────────────────────────────────────
    # SUMMARY WATERFALL
    # ────────────────────────────────────────────────────────────────
    hdr("SUMMARY WATERFALL")
    waterfall = [
        ("Stage_3_Cleaning (pre-rules)",                     cp2_pre),
        ("decisions_aggregate.csv, action=='keep' only",     cp3_rule_only),
        ("01_unioned (rule+union+manual+identity applied)",  cp4_post_union),
        ("02_assembled/agg_full_moments.parquet",            cp5_assembled),
        ("Stage_5_Themes final numbered inventory",          cp6_final),
    ]
    prev = None
    for label, val in waterfall:
        if val is None:
            print(f"  {label:<52} (not available)")
            continue
        delta = "" if prev is None else f"   ({val - prev:+,})"
        print(f"  {label:<52} {val:>6,}{delta}")
        prev = val

    print(f"""
  CONCLUSION
  ----------
  '2,437' is what survives if you stop after rules S/R0-R6 alone, and it
  is never written to disk as such. The real pipeline keeps going:
  Stage_4_Assembly/01_apply_union.ipynb applies the union of aggregate-
  and panel-side rule failures, 13 manually-dropped factors, and an
  identity-enforcement pass between the two pipelines, all before
  Stage_4_Assembly/07_drop_binaries.ipynb removes the five regime
  indicators. None of that second pass appears in the "Exclusion Rules
  and Results" section, which is why 2,437 and 1,699 don't reconcile as
  currently written.

  Suggested fix for the write-up: either state explicitly that a further
  union/manual/identity-enforcement/binary-drop pass follows (with the
  real numbers from Checkpoint 4/5 above), or move the "1,699-factor,
  331-subtheme" summary out of this section entirely and place it next
  to wherever that later pass is actually described.
""")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--root", type=Path, default=None,
        help="Path to Data/Data_Collection/Final. If omitted, tries a few "
             "likely relative locations.",
    )
    args = parser.parse_args()

    if args.root is not None:
        root = args.root
    else:
        candidates = [
            Path("Data/Data_Collection/Final"),
            Path("../Data/Data_Collection/Final"),
            Path("../../Data/Data_Collection/Final"),
            Path("../../../Data/Data_Collection/Final"),
        ]
        root = next((c for c in candidates if c.exists()), candidates[0])

    if not root.exists():
        sys.exit(
            f"Could not find {root} -- pass --root explicitly, e.g.\n"
            f"  python {Path(__file__).name} "
            f'--root "C:/.../Data/Data_Collection/Final"'
        )

    main(root)

usage: ipykernel_launcher.py [-h] [--root ROOT]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\Henry\AppData\Roaming\jupyter\runtime\kernel-v3a1d4aba12aab33f063a9233fe331f305b2ad8e04.json


SystemExit: 2

c:\Users\Henry\anaconda3\envs\diss\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
